# Welfare Bot - Data preprocessing

Tässä notebookissa valmistan hyvinvointidatan koneoppimismalleja varten.

Preprocessing-vaiheessa:
- käsittelen puuttuvat arvot
- tarkistan datan laadun
- valitsen koneoppimismallille tärkeät muuttujat
- skaalan numeeriset muuttujat

Tavoitteena on valmistella data mahdollisimman hyvin myöhempiä koneoppimisanalyysejä varten.

In [ ]:
# kirjastojen lataaminen
import pandas as pd # pandas auttaa taulukkomuotoisen datan käsittelyssä
import numpy as np # numpy tarjoaa tehokkaita työkaluja numeeriseen laskentaan
from sklearn.preprocessing import StandardScaler # StandardScaler auttaa datan normalisoinnissa


In [2]:
# datan lataaminen
data = pd.read_csv('../data/welfare_bot_metrics_missing.csv')



## Puuttuvien arvojen tarkastelu
Ennen koneoppimismallien käyttöä tarkistan datan puuttuvat arvot.


In [3]:
data.isnull().sum() # tarkistaa puuttuvat arvot jokaisessa sarakkeessa, koska puuttuvat arvot 
                    # voivat vaikuttaa koneoppimismallin toimintaan.

date                  0
user_name             0
email                 0
mood_score           45
sleep_score          33
food_score           39
hydration_score      57
medication_score     16
social_score         39
overall_score       202
risk_level          202
dtype: int64

## Puuttuvien arvojen käsittely

Tässä vaiheessa käsittelen datan puuttuvat arvot.
Koneoppimismallit eivät yleensä pysty käyttämään tyhjiä arvoja, joten ennen Isolation Forest mallia minun täytyy korjata puuttuvat tiedot.
Täytän numeeristen sarakkeiden puuttuvat arvot mediaanilla.
Mediaani on hyvä vaihtoehto, koska se ei muutu yhtä helposti poikkeavien arvojen takia kuin keskiarvo.

In [4]:
# valitaan numeeriset sarakkeet
numeric_columns = [
    "mood_score",
    "sleep_score",
    "food_score",
    "hydration_score",
    "medication_score",
    "social_score",
    "overall_score"
]

# täytetään puuttuvat arvot mediaanilla
data[numeric_columns] = data[numeric_columns].fillna(
    data[numeric_columns].median()
)

In [5]:
data.isnull().sum() # tarkistaa uudestaan, että kaikki puuttuvat arvot on täytetty, jotta voin olla varma, että data on valmis  


date                  0
user_name             0
email                 0
mood_score            0
sleep_score           0
food_score            0
hydration_score       0
medication_score      0
social_score          0
overall_score         0
risk_level          202
dtype: int64

## Feature selection
Tässä vaiheessa valitsen koneoppimismallille käytettävät muuttujat.
Valitsin hyvinvointiin liittyvät numeeriset mittarit, koska niiden avulla voidaan tunnistaa tilanteita, joissa käyttäjän hyvinvointi poikkeaa normaalista.
Mukana ovat: mieliala, uni, ruokailu, nesteytys, lääkitys, sosiaalinen aktiivisuus, kokonaisvaltainen hyvinvointi
En ottanut mukaan sarakkeita: user_name, email, date ja risk_level, koska ne eivät suoraan vaikuta hyvinvointianalyysiin

In [6]:
# valitaan featuret mallia varten
features = [
    "mood_score",
    "sleep_score",
    "food_score",
    "hydration_score",
    "medication_score",
    "social_score",
    "overall_score"
]

X = data[features] # luodaan uusi muuttuja X, joka sisältää vain valitut featuret. 

Käytän StandardScaler-menetelmää, koska hyvinvointimittarit ovat eri asteikoilla. Tämä näkyy 'describe()'-analyysissä: 'hydration_score' on välillä 0–1; 'social_score' on välillä 0–5; 'overall_score' on noin välillä 4–10
Skaalaus auttaa koneoppimismallia käsittelemään kaikkia muuttujia tasapuolisemmin.

In [8]:
from sklearn.preprocessing import StandardScaler # StandardScaler auttaa datan normalisoinnissa
# Luodaan scaler-objekti
scaler = StandardScaler()
# Skaalataan featuret
X_scaled = scaler.fit_transform(X)

## Datan tallentaminen

Tallennan preprocessing-vaiheen jälkeen käsitellyn datan myöhempiä koneoppimisanalyysejä varten.

Tämän jälkeen samaa käsiteltyä dataa voidaan käyttää:
- Isolation Forest -analyysissä
- lineaarisessa regressiossa
- muissa koneoppimismalleissa

In [12]:
# tallennetaan käsitelty data

data.to_csv("../data/welfare_bot_metrics_cleaned.csv", index=False)


In [13]:
# tarkistetaan tallennettu data

cleaned_data = pd.read_csv("../data/welfare_bot_metrics_cleaned.csv")
cleaned_data.head()

,date,user_name,email,mood_score,sleep_score,food_score,hydration_score,medication_score,social_score,overall_score,risk_level
0,2025-01-01,Aino Mäkinen,aino.makinen@demo.fi,4.40,3.94,2.99,1.00,0.88,3.38,8.890,low
1,2025-01-02,Aino Mäkinen,aino.makinen@demo.fi,4.83,4.31,2.66,0.96,0.86,3.27,8.980,low
2,2025-01-03,Aino Mäkinen,aino.makinen@demo.fi,4.30,3.24,2.28,0.80,0.82,3.66,7.670,low
3,2025-01-04,Aino Mäkinen,aino.makinen@demo.fi,3.84,3.44,3.00,0.79,0.91,2.79,7.765,NaN
4,2025-01-05,Aino Mäkinen,aino.makinen@demo.fi,3.99,4.05,2.46,0.94,0.85,3.36,8.250,low


## Preprocessing-vaiheen yhteenveto

Preprocessing-vaiheessa:
- tarkistin datan laadun
- käsittelin puuttuvat arvot
- valitsin koneoppimismallille tärkeät muuttujat
- skaalasin numeeriset featuret StandardScaler-menetelmällä

Näiden vaiheiden jälkeen data oli valmis koneoppimismallien käyttöä varten.